# semanticdrift on Google Colab

Runtime → Change runtime type → **GPU**. Keep this tab in the foreground while models download and while `pipeline` runs.

Do not put `.github_token` in a cell. Private clone: Colab Secrets → `GITHUB_TOKEN`.


In [ ]:
!apt-get -qq update
!apt-get -qq install -y spin gcc g++ make git
!spin -V | head -n 1


In [ ]:
REPO = "colorfulelina/semanticdrift"

import os
from pathlib import Path

dest = Path("/content/semanticdrift")
if not (dest / "src/semanticdrift").is_dir():
    url = REPO if REPO.startswith("http") else f"https://github.com/{REPO}.git"
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = os.environ.get("GITHUB_TOKEN") or None
    if token and "github.com" in url:
        url = url.replace("https://", f"https://{token}@", 1)
    !git clone --depth 1 {url} /content/semanticdrift

%cd /content/semanticdrift
!bash scripts/colab_setup.sh


In [ ]:
%cd /content/semanticdrift
!python -m semanticdrift check-promela
!python -m semanticdrift prove-promela


## Language models + pipeline

Install Ollama, pull Qwen then DeepSeek (validate needs both). First run **one** protocol. If that finishes, run all ten. If the session disconnects, start again from Ollama serve (models stay on disk only until the VM is recycled).


In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time, urllib.request

subprocess.Popen(["ollama", "serve"])
for _ in range(30):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/", timeout=1)
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("ollama serve did not start")

!ollama pull qwen2.5-coder:7b
!ollama pull deepseek-coder-v2:lite
!ollama list


In [ ]:
%cd /content/semanticdrift
!python -m semanticdrift pipeline --name peterson --requirement precise


In [ ]:
%cd /content/semanticdrift
!python -m semanticdrift pipeline --all
!python -m semanticdrift metrics
